# Scraping AWOIAF Character Biographies
Produces `characters_bio.csv` with columns: `name`, `ID`, `bio`

- **`bio`**: The full narrative text from the character's wiki page (their story history), with all recognised character names replaced by their canonical `ID` slug.
- This makes the bio machine-readable for downstream tasks like relationship sentiment extraction.

Requires `characters.csv` from the original `scrape_characters.ipynb` (or will re-scrape the list if not found).

In [1]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import re
import os
from urllib.parse import quote, unquote
from concurrent.futures import ThreadPoolExecutor

## Setup

In [2]:
BASE   = 'https://awoiaf.westeros.org'
PREFIX = '/index.php/'

headers = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
}

session = requests.Session()
session.headers.update(headers)

## 1. Load or scrape the character list
Reuses `characters.csv` from the original notebook if it exists.

In [4]:
CHARACTERS_CSV = 'characters.csv'

def scrape_character_list():
    list_page = session.get(BASE + PREFIX + 'List_of_characters', timeout=20)
    print(f'List page status: {list_page.status_code}')
    soup = BeautifulSoup(list_page.content, 'html.parser')
    content = soup.find('div', class_='mw-parser-output')
    characters = []
    seen_ids = set()
    for li in content.find_all('li'):
        a = li.find('a')
        if not a:
            continue
        href = a.get('href', '')
        if not href.startswith(PREFIX):
            continue
        if 'redlink=1' in href or 'action=' in href:
            continue
        slug = href[len(PREFIX):].split('#', 1)[0]
        if not slug or slug.startswith('Special:'):
            continue
        slug = unquote(slug)
        if slug in seen_ids:
            continue
        seen_ids.add(slug)
        name = a.get_text(strip=True)
        if not name:
            continue
        characters.append({'name': name, 'ID': slug})
    return characters

if os.path.exists(CHARACTERS_CSV):
    char_df = pd.read_csv(CHARACTERS_CSV)
    print(f'Loaded {len(char_df)} characters from {CHARACTERS_CSV}')
else:
    print(f'{CHARACTERS_CSV} not found — scraping character list...')
    chars = scrape_character_list()
    char_df = pd.DataFrame(chars, columns=['name', 'ID'])
    char_df.to_csv(CHARACTERS_CSV, index=False)
    print(f'Saved {len(char_df)} characters to {CHARACTERS_CSV}')

characters = char_df.to_dict('records')
print(characters[:5])

Loaded 3690 characters from characters.csv
[{'name': 'A certain man', 'ID': 'A_certain_man'}, {'name': 'Abelar Hightower', 'ID': 'Abelar_Hightower'}, {'name': 'Abelon', 'ID': 'Abelon'}, {'name': 'Addam of Duskendale', 'ID': 'Addam_of_Duskendale'}, {'name': 'Addam Frey', 'ID': 'Addam_Frey'}]


## 2. Build name → ID replacement map

We build a mapping from every character's **display name** to their **ID slug**.
Names are sorted longest-first so that longer/more-specific names are replaced before shorter substrings
(e.g. "Jon Connington" before "Jon").

In [5]:
# name -> ID, sorted longest name first to avoid partial replacements
name_to_id = {row['name']: row['ID'] for row in characters}

# Pre-compile a single regex that matches any character name (word-boundary anchored)
# Sorted longest-first so "Aegon Targaryen" matches before "Aegon"
sorted_names = sorted(name_to_id.keys(), key=len, reverse=True)
pattern = re.compile(
    '|'.join(re.escape(n) for n in sorted_names)
)

def replace_names_with_ids(text):
    """Replace all character display names in `text` with their canonical ID slug."""
    return pattern.sub(lambda m: name_to_id[m.group(0)], text)

print(f'Replacement map built for {len(sorted_names)} character names.')
# Quick sanity check
test = 'Eddard Stark and Jon Snow rode north.'
print('Test replacement:', replace_names_with_ids(test))

Replacement map built for 3562 character names.
Test replacement: Eddard_Stark and Jon_Snow rode north.


## 3. Helpers: URL utilities & bio extraction

In [6]:
def slug_to_url(slug):
    return BASE + PREFIX + quote(slug, safe="_/(),'")


# Sections that are NOT biography / history — skip these entirely
SKIP_SECTIONS = {
    'references', 'notes', 'external links', 'see also',
    'navigation', 'appendix', 'citations', 'gallery',
    'quotes', 'tv adaptation', 'behind the scenes',
}

# Sections that ARE the biography/history — prefer these if present
BIO_SECTIONS = {
    'history', 'biography', 'background', 'recent events',
    'life', 'character and appearance',
}


def extract_bio(soup):
    """
    Extract the narrative biography text from a character page.

    Strategy:
    1. Find the article body (mw-parser-output).
    2. Walk headings to identify sections.
    3. Collect paragraphs from biography/history sections.
       If none are found, fall back to ALL non-skip-section paragraphs.
    4. Strip wiki-internal boilerplate (citation markers like [1], [2]).
    """
    content = soup.find('div', class_='mw-parser-output')
    if content is None:
        return ''

    # Collect all top-level elements in document order
    elements = list(content.children)

    bio_paragraphs = []
    all_paragraphs = []
    current_section = None   # None = lead section (before first heading)
    in_bio_section = False

    for el in elements:
        if not hasattr(el, 'name') or el.name is None:
            continue  # skip NavigableString nodes (whitespace)

        # Track which section we're in
        if el.name in ('h2', 'h3', 'h4'):
            heading_text = el.get_text(' ', strip=True).lower()
            # Strip edit-section spans that MediaWiki injects
            heading_text = re.sub(r'\[edit\]', '', heading_text).strip()
            current_section = heading_text
            in_bio_section = any(kw in current_section for kw in BIO_SECTIONS)
            continue

        # Skip non-paragraph elements (tables = infoboxes, navboxes; divs = toc/notices)
        if el.name not in ('p',):
            continue

        text = el.get_text(' ', strip=True)
        if not text or len(text) < 30:   # skip stub/empty paras
            continue

        # Remove citation markers like [1], [2], [nb 1]
        text = re.sub(r'\[\d+\]', '', text)
        text = re.sub(r'\[nb \d+\]', '', text)
        text = text.strip()

        if not text:
            continue

        # Skip-section check
        if current_section and any(kw in current_section for kw in SKIP_SECTIONS):
            continue

        all_paragraphs.append(text)

        # Lead section (before first heading) and known bio sections
        if current_section is None or in_bio_section:
            bio_paragraphs.append(text)

    result = bio_paragraphs if bio_paragraphs else all_paragraphs
    return ' '.join(result)

## 4. Per-character scrape

In [7]:
def scrape_bio(row):
    """
    Fetch one character page and return {'name', 'ID', 'bio'}.
    The bio has all recognised character names replaced with their ID slug.
    """
    name, slug = row['name'], row['ID']
    try:
        resp = session.get(slug_to_url(slug), timeout=20)
    except requests.RequestException as e:
        print(f'  ! request failed for {slug}: {e}')
        return {'name': name, 'ID': slug, 'bio': ''}

    if resp.status_code != 200:
        print(f'  ! HTTP {resp.status_code} for {slug}')
        return {'name': name, 'ID': slug, 'bio': ''}

    soup = BeautifulSoup(resp.content, 'html.parser')
    raw_bio = extract_bio(soup)
    bio_with_ids = replace_names_with_ids(raw_bio)

    return {'name': name, 'ID': slug, 'bio': bio_with_ids}

## 5. Run the scrape
Parallelised across 8 threads, checkpoints every 100 rows.

In [8]:
OUT     = 'characters_bio.csv'
COLUMNS = ['name', 'ID', 'bio']
WORKERS = 8

rows = []
with ThreadPoolExecutor(max_workers=WORKERS) as executor:
    for i, row in enumerate(executor.map(scrape_bio, characters), 1):
        rows.append(row)
        if i % 50 == 0:
            print(f'  {i}/{len(characters)}')
        if i % 100 == 0:
            pd.DataFrame(rows, columns=COLUMNS).to_csv(OUT, index=False)

pd.DataFrame(rows, columns=COLUMNS).to_csv(OUT, index=False)
print(f'\nSaved {len(rows)} rows to {OUT}')

  50/3690
  100/3690
  150/3690
  200/3690
  250/3690
  300/3690
  350/3690
  400/3690
  450/3690
  500/3690
  550/3690
  600/3690
  650/3690
  700/3690
  750/3690
  800/3690
  850/3690
  900/3690
  950/3690
  1000/3690
  1050/3690
  1100/3690
  1150/3690
  1200/3690
  1250/3690
  1300/3690
  1350/3690
  1400/3690
  1450/3690
  1500/3690
  1550/3690
  1600/3690
  1650/3690
  1700/3690
  1750/3690
  1800/3690
  1850/3690
  1900/3690
  1950/3690
  2000/3690
  2050/3690
  2100/3690
  2150/3690
  2200/3690
  2250/3690
  2300/3690
  2350/3690
  2400/3690
  2450/3690
  2500/3690
  2550/3690
  2600/3690
  2650/3690
  2700/3690
  2750/3690
  2800/3690
  2850/3690
  2900/3690
  2950/3690
  3000/3690
  3050/3690
  3100/3690
  3150/3690
  3200/3690
  3250/3690
  3300/3690
  3350/3690
  3400/3690
  3450/3690
  3500/3690
  3550/3690
  3600/3690
  3650/3690

Saved 3690 rows to characters_bio.csv


## 6. Sanity check

In [9]:
df = pd.read_csv(OUT)
print(f'Shape: {df.shape}')
print(f'Empty bios: {(df["bio"] == "").sum()}')
print(f'Median bio length (chars): {df["bio"].str.len().median():.0f}')
print()

# Inspect a well-known character
sample = df[df['name'] == 'Eddard Stark']
if not sample.empty:
    print('--- Eddard Stark bio (first 800 chars) ---')
    print(sample.iloc[0]['bio'][:800])

Shape: (3690, 3)
Empty bios: 0
Median bio length (chars): 244

--- Eddard Stark bio (first 800 chars) ---
Eddard_Stark , also called "Ned_(ferryman)" , is the head of House Stark , Lord of Winterfell , and Warden of the North . He is a close friend to King Robert_I_Baratheon , with whom he was raised. Eddard is one of the major POV characters in A Song of Ice and Fire . In the television adaptation Game of Thrones , Eddard is portrayed by Sean Bean , Sebastian Croft (as a child), and Robert Aramayo (as a young man).


---
## Notes for downstream sentiment / relationship extraction

The `bio` column now contains prose where every recognised character name has been
replaced with their unique `ID` slug (e.g. `Eddard_Stark`, `Jon_Snow`).

**Suggested next steps:**

1. **Sentence-level co-occurrence**: Split each bio into sentences, find pairs of IDs that appear together, and label the sentence as positive/negative/neutral.
2. **LLM extraction**: For each character, pass their bio to a model with a prompt like:
   > *"For each character ID mentioned in this biography, classify the subject's relationship to them as: positive / negative / neutral. Output JSON."*
3. **Build an edge list**: Combine all (source_ID, target_ID, sentiment) triples into a signed graph for network analysis.

The ID substitution ensures that even highly ambiguous names ("Jon", "Aegon") map to
unambiguous nodes in the graph.